# Raining Hoops — WNBA 2026 Data Bootstrap

This notebook uses the versioned WNBA Stats release published by SportsDataverse. The release is built from official `stats.wnba.com` endpoints and contains both `regular-season` and `playoffs` player-game logs for 2026. It avoids direct access to `stats.nba.com` from Colab.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/samyak0407/raininghoops.git"
REPO_DIR = "/content/raininghoops"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
%cd /content/raininghoops
print("Repository ready:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed.")

In [ ]:
from src.data.wnba_release_client import WNBAStatsReleaseClient
from src.data.normalize import normalize_player_game_logs
from src.data.quality import assert_valid_player_game_logs, validate_player_game_logs

client = WNBAStatsReleaseClient()
regular = client.player_game_logs(2026, season_type="Regular Season")
playoffs = client.player_game_logs(2026, season_type="Playoffs")
raw = __import__("pandas").concat([regular, playoffs], ignore_index=True)
print("Regular-season rows:", f"{len(regular):,}")
print("Playoff rows:", f"{len(playoffs):,}")
print("Combined rows:", f"{len(raw):,}")

normalized = normalize_player_game_logs(raw)
assert_valid_player_game_logs(normalized)
quality = validate_player_game_logs(normalized)
print("Players:", f"{normalized['player_id'].nunique():,}")
print("Games:", f"{normalized['game_id'].nunique():,}")
print("Date range:", normalized['game_date'].min(), "->", normalized['game_date'].max())
print("Season types:", normalized['season_type'].value_counts().to_dict())
display(normalized.head())
print("Quality report:")
print(quality)

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

output_dir = Path("data/raw/player_game_logs")
output_dir.mkdir(parents=True, exist_ok=True)
output = output_dir / "wnba_2026_all_player_game_logs.parquet"
normalized.to_parquet(output, index=False)
manifest = {
    "league": "WNBA",
    "season": "2026",
    "season_types": ["Regular Season", "Playoffs"],
    "source": "WNBA Stats official endpoints via SportsDataverse versioned release",
    "source_url": "https://github.com/sportsdataverse/sportsdataverse-data/releases/tag/wnba_stats_player_game_logs",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "rows": len(normalized),
    "players": int(normalized["player_id"].nunique()),
    "games": int(normalized["game_id"].nunique()),
    "date_min": str(normalized["game_date"].min()),
    "date_max": str(normalized["game_date"].max()),
    "quality": quality,
    "output": str(output),
}
manifest_path = output.with_suffix(".json")
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print("Saved:", output)
print("Manifest:", manifest_path)

## Next

With the verified player-game dataset loaded, Raining Hoops can move immediately into rolling opportunity, role shifts, teammate availability, matchup context, prediction, and evaluation. StatPick remains a separate market-intelligence input.